In [1]:
# 1. 数据基础库
%pip install numpy pandas

# 2. 可视化库
%pip install matplotlib seaborn

# 3. 机器学习库
%pip install scikit-learn

# 4. 统计分析库
%pip install statsmodels scipy

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'd:\modeling\.venv\Scripts\python.exe -m pip install --upgrade pip' command.


Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'd:\modeling\.venv\Scripts\python.exe -m pip install --upgrade pip' command.


Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'd:\modeling\.venv\Scripts\python.exe -m pip install --upgrade pip' command.


Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'd:\modeling\.venv\Scripts\python.exe -m pip install --upgrade pip' command.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# 统计分析库
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox

# 机器学习库
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 设置中文显示和图形样式
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
print("=" * 60)
print("         时间序列预测ARIMA模型 - 股票价格预测案例")
print("=" * 60)

第一步：数据生成与加载

In [ ]:
def generate_stock_data(start_date='2025-01-01', periods=100,initial_price=100):
    """
    生成模拟股票价格数据
    包含趋势、波动率和随机游走特征
    """
    np.random.seed(42)  # 保证结果可复现

    # 创建日期索引
    dates = pd.date_range(start=start_date, periods=periods, frq='D')

    # 生成价格序列：随机游走+微弱趋势+波动率变化
    returns = np.random.normal(0.001, 0.02,periods)  #日收益率
    trend = np.linspace(0,0.5,periods)/periods  #微弱上升趋势
    volatility = 1 + 0.3 * np.sin(np.linspace(0, 4*np.pi, periods))  #波动率周期变化

    returns = returns + trend
    returns = returns * volatility

    # 计算累积价格
    prices = [initial_price]
    for r in returns[1:]:
        prices.append(prices[-1] * (1+r))

    # 创建dataframe
    df = pd.DataFrame({
        'date':dates,
        'price':prices,
        'returns':[0] + list(np.diff(np.log(prices)))  # 对数收益率
    })
    df.set_index('date',inplace=True)

    return df


# 生成数据
print("\n1.数据生成与加载")
print('-'*30)
stock_data = generate_stock_data()
print(f"数据区间：{stock_data.index[0].strftime('%Y-%m-%d')}至{stock_data.index[-1].strftime('%Y-%m-%d')}")
print(f"数据点数: {len(stock_data)}")
print(f"价格范围: {stock_data['price'].min():.2f} - {stock_data['price'].max():.2f}")

# 显示基本统计信息
print("\n数据基本统计:")
print(stock_data.describe())

第二步 数据探索与可视化

In [ ]:
print("\n2.数据探索与可视化")
print('-'*30)

# 创建综合图标
fig, axes = plt.subplot(2, 2, figsize=(15,10))

# 价格时序图
axes[0,0].plot(stock_data.index,stock_data['price'],color='steelblue',linewidth=1.5)
axes.set_title('股票价格时序图',fontsize=14,fontweight='bold')
axes.set_ylabel('价格（元）')
axes.grid(True, alpha=0.3)

# 收益率时序图
